# **HiddenGems:** Analysing NBA contract value with machine learning. 

**Module:** Final Year Project<br>
**Student:** Oisin Webb (K00293591)<br>

## **Profiling**
This notebook explores the 7 raw tables loaded into PostgreSQL to find data quality issues before cleaning. The raw tables are left untouched, and cleaned versions are saved separately.

## **Cross Refenecing Website** :
**Basketball Refence:** https://www.basketball-reference.com/

In [1]:
# Author: Oisin Webb

from sqlalchemy import create_engine
from db_config import DB_URL

import pandas as pd
import numpy as np

In [2]:
# Author: Oisin Webb
# Database connection

engine = create_engine(DB_URL)

## Profiling Players Table


In [3]:
# Author: Oisin Webb
# Reading in the players table in the .raw schema.

players_df = pd.read_sql("SELECT * FROM raw.players", engine)
players_df.shape

(5477, 10)

## What players_df.shape shows me
- 5477 rows in the table.
- 10 columns.

In [4]:
# Author: Oisin Webb
players_df.head(10)

,player_id,player_name,full_name,position,shoots,height_cm,weight_lbs,nickname,career_length_years,jersey_numbers
0,stirtbe01,Bennett Stirtz,Bennett Carl Stirtz,Point Guard,Right,193.0,186.0,NaN,NaN,NaN
1,maraad01,Aday Mara,Aday Mara,Center,Right,221.0,260.0,NaN,NaN,NaN
2,naberbo01,Bob Naber,Bob Naber,Forward,Right,190.0,185.0,NaN,1.0,{14}
3,traylro01,Robert Traylor,Robert Traylor,Center and Power Forward,Right,203.0,284.0,Tractor Traylor,7.0,"{54,34,32}"
4,volkoal01,Alexander Volkov,Alexander Volkov,Center,Right,208.0,218.0,Sasha,2.0,{8}
5,clarkco01,Coty Clarke,Coty Clarke,Small Forward,Right,201.0,232.0,NaN,1.0,{63}
6,flemiki01,Kingston Flemings,Kingston Flemings,Point Guard,Right,193.0,183.0,NaN,NaN,NaN
7,philola01,Labaron Philon Jr.,Labaron Dewayne Philon Jr.,Point Guard,Right,193.0,176.0,NaN,NaN,NaN
8,radzira01,Ray Radziszewski,Ray Radziszewski,Forward,Right,196.0,210.0,NaN,1.0,{19}
9,boothca01,Calvin Booth,Calvin Booth,Center,Left,211.0,230.0,NaN,10.0,{52}


## What players_df.head(10) shows me

- There is two name columns only one will be needed probably `player_name` as I have another table that fixes discrepancies.
- Players can have more then one position.
- `shoots` column is for what hand the player uses to shoot
- `nickname` has many rows with "NaN" value this colunm will be dropped in cleaning.
- `career_length_years` some players with "NaN" are players that have just drafted in the 2026 NBA draft. Will investigate more further down.
- `jersey_number` also on the rows of players that have been just drafted in the 2026 NBA draft. Also one player can many different jersey numbers.

In [14]:
# Author: Oisin Webb
players_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5477 entries, 0 to 5476
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   player_id            5477 non-null   str    
 1   player_name          5477 non-null   str    
 2   full_name            5476 non-null   str    
 3   position             5476 non-null   str    
 4   shoots               5476 non-null   str    
 5   height_cm            5471 non-null   float64
 6   weight_lbs           5473 non-null   float64
 7   nickname             2139 non-null   str    
 8   career_length_years  4685 non-null   float64
 9   jersey_numbers       5373 non-null   str    
dtypes: float64(3), str(7)
memory usage: 795.8 KB


## What players_df.info() shows
- All the data types look acceptable, except `jersey_numbers`, which is stored as a string. This is because a player can wear several numbers over their career, so the column holds more than one value per player.

In [6]:
# Author: Oisin Webb
# Description of all the numeric columns

players_df.describe()

,height_cm,weight_lbs,career_length_years
count,5471.000000,5473.000000,4685.000000
mean,198.171632,209.665083,4.998506
std,9.139682,25.852960,4.470393
min,160.000000,114.000000,1.000000
25%,190.000000,190.000000,1.000000
50%,198.000000,210.000000,3.000000
75%,206.000000,225.000000,8.000000
max,231.000000,360.000000,22.000000


## What players_df.describe() shows me 
- 231 cm or 7'7 is the max in the `height_cm` it's not unheard of in the NBA.
- 360 pounds is the max in the `weight_lbs`.
- `career_length_years` min years is 1 which is understandable and max is 22 years.
### All above have will be cross-referenced using basketball reference to verify stats are correct.

In [7]:
# Author: Oisin Webb
players_df[players_df["height_cm"] == 231][["player_name", "career_length_years","height_cm","weight_lbs"]]

,player_name,career_length_years,height_cm,weight_lbs
1972,Gheorghe Mureșan,6.0,231.0,303.0
4902,Manute Bol,10.0,231.0,200.0


- `Gheorghe Mureșan` and `Manute Bol` were both indeed 231 cm or 7'7 according to basketball reference.

In [8]:
# Author: Oisin Webb
players_df[players_df["weight_lbs"] == 360][["player_name","career_length_years","height_cm","weight_lbs"]]

,player_name,career_length_years,height_cm,weight_lbs
5448,Sim Bhullar,1.0,226.0,360.0


- `Sim Bhullar` was indeed 360 pounds according to basketball reference.

In [9]:
# Author: Oisin Webb
players_df[players_df["career_length_years"] == 22][["player_name","career_length_years","height_cm","weight_lbs"]]

,player_name,career_length_years,height_cm,weight_lbs
849,Vince Carter,22.0,198.0,220.0


- `Vince Carter` did indeed player 22 years in the NBA according to basketball reference.

In [10]:
# Author: Oisin Webb
players_df.isna().sum()

player_id                 0
player_name               0
full_name                 1
position                  1
shoots                    1
height_cm                 6
weight_lbs                4
nickname               3338
career_length_years     792
jersey_numbers          104
dtype: int64

## What players_df.isna().sum() shows me

- `player_name` has no missing values, so it is used instead of `full_name` (1 missing).
- `position` and `shoots` have 1 missing each, and `height_cm` (6) and `weight_lbs` (4) a few more. These will be filled by hand from Basketball Reference.
- `nickname` (3,338), `career_length_years` (792) and `jersey_numbers` (104) have many gaps and aren't needed for the analysis, so they will be dropped.

In [17]:
# Author: Oisin Webb
players_df.nunique()

player_id              5477
player_name            5428
full_name              5427
position                 44
shoots                    2
height_cm                28
weight_lbs              149
nickname               1932
career_length_years      22
jersey_numbers         2585
dtype: int64

## What players_df.nunique() shows me

- `player_id` has 5,477 unique values, one per row, so it uniquely identifies each player and is used to link tables.
- `player_name` has 5,428 unique values, so 49 names are shared by different players. Names alone can't identify a player.
- `position` has 44 different values, more than basketball's 5 positions so these are counting the players that can or have played more then one position in thier career.
- `shoots` has 2 values (left and right), as expected.